# 01: Data exploration

Loads the dataset and runs the Phase 1 profiler. Logic lives in `src/ai_experiment_lab/data/`; this notebook only calls it.

**Before running:** upload the dataset CSV to `MyDrive/ai-experiment-lab/data/` and set `DATASET_FILE` below.

In [4]:
# --- AI Experiment Lab: standard Colab setup cell ---------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/sudhanshu230230/AI_Experiment_Lab.git"   # <-- edit
REPO_DIR = "/content/ai-experiment-lab"
DRIVE_ROOT = "/content/drive/MyDrive/ai-experiment-lab"
EXTRAS = "ml,tracking"          # pip extras from pyproject.toml

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/mlflow", exist_ok=True)

# For a private repo, store a token in Colab Secrets as GITHUB_TOKEN (never paste it here).
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[{EXTRAS}]"], check=True)

# An editable install adds a .pth file, which a running kernel does not re-read.
# Put src/ on sys.path so the import works in this session without a restart.
src_dir = f"{REPO_DIR}/src"
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Tracking location: persistent Drive storage (see docs/decisions/ADR-001-mlflow-storage.md).
os.environ["MLFLOW_TRACKING_URI"] = f"file:{DRIVE_ROOT}/mlflow"
os.environ["AEL_DATA_DIR"] = f"{DRIVE_ROOT}/data"

commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code version (commit):", commit)
print("Data dir:", os.environ["AEL_DATA_DIR"])

import ai_experiment_lab
print("ai_experiment_lab", ai_experiment_lab.__version__, "imported OK")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code version (commit): c8b1e93
Data dir: /content/drive/MyDrive/ai-experiment-lab/data
ai_experiment_lab 0.1.0 imported OK


In [5]:
from pathlib import Path
from ai_experiment_lab.data import load_csv, profile_dataframe, format_report
from ai_experiment_lab.utils.config import get_data_dir, load_config

DATASET_FILE = "Telco-Customer-Churn.csv"     # <-- file name in your Drive data folder
TARGET = "Churn"

df, info = load_csv(Path(get_data_dir()) / DATASET_FILE)
print(info)          # record sha256 in the phase report (dataset version)
df.head()

DatasetInfo(name='Telco-Customer-Churn', path='/content/drive/MyDrive/ai-experiment-lab/data/Telco-Customer-Churn.csv', sha256='88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a', n_rows=7043, n_cols=21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
profile = profile_dataframe(df, target=TARGET)
print(format_report(profile))

DATASET PROFILE
Rows: 7,043   Columns: 21
Numeric features: 3   Categorical features: 17
Missing cells: 0.01%   Duplicate rows: 0
Columns with missing or blank values:
  - TotalCharges: 0 NaN, 11 blank
Target 'Churn': 2 classes, distribution {'No': 0.7346, 'Yes': 0.2654}
  Minority fraction: 26.54%   Imbalance ratio: 2.77:1


In [7]:
# Raw dtypes and a closer look at anything the report flagged
display(df.dtypes.to_frame("dtype").T)
for col in profile["numeric_like_text_features"]:
    print(col, "-> non-numeric values:", df[col][df[col].astype(str).str.strip().eq("")].shape[0], "blank")

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
dtype,object,object,int64,object,object,int64,object,object,object,object,...,object,object,object,object,object,object,object,float64,object,object


TotalCharges -> non-numeric values: 11 blank


## Observations

Write findings here, then copy the conclusions into `docs/phases/phase-01-*.md`:

- Missing / blank values:
- Columns needing type conversion:
- Identifier columns to exclude:
- Class balance:
- Surprises: